In [6]:
import pandas as pd
df = pd.read_csv('/home/honglinbao/chain_of_hints/diff.csv')
df['distance'].mean()

0.3419597241157377

In [ ]:
import pandas as pd
df = pd.read_parquet('/net/scratch/honglinbao/siyang/coh_2015.parquet')
df

,FieldOfStudyName,id,recovered_abstract,embedding
0,spiral,2869521522,the utility model discloses a radiating type s...,"[-0.3100000023841858, 0.07000000029802322]"
1,fertilizer,2858879474,the invention discloses a plant-source insect ...,"[-0.14000000059604645, -0.05000000074505806]"
2,potential flow,2605022811,in this work we have obtained exact analytical...,"[0.05999999865889549, 0.14000000059604645]"
3,right main bronchus,2463081776,malignant peripheral nerve sheath tumors (mpns...,"[0.15000000596046448, 0.05999999865889549]"
4,causality,3121301771,this paper adopts a nonparametric quantile cau...,"[0.009999999776482582, -0.14000000059604645]"
...,...,...,...,...
1462051,phy,terms196,mie scattering,"[0.07000000029802322, 0.11999999731779099]"
1462052,phy,terms197,rydberg blockade,"[0.05999999865889549, 0.019999999552965164]"
1462053,phy,terms198,fabry–perot,"[0.009999999776482582, -0.10000000149011612]"
1462054,phy,terms199,raman spectroscopy,"[0.20999999344348907, 0.1599999964237213]"


In [3]:
import pandas as pd
df_non_term = df[~df['id'].astype(str).str.startswith('terms')].copy()
df_non_term['abstract_word_count'] = df_non_term['recovered_abstract'].fillna('').apply(lambda x: len(x.split()))
print(df_non_term['abstract_word_count'].describe())
print("最小词数：", df_non_term['abstract_word_count'].min())

count    1.461856e+06
mean     1.736439e+02
std      9.155522e+01
min      1.000000e+01
25%      1.200000e+02
50%      1.630000e+02
75%      2.140000e+02
max      1.609000e+03
Name: abstract_word_count, dtype: float64
最小词数： 10


In [1]:
import pandas as pd
import ast
df = pd.read_parquet('/net/scratch/honglinbao/coh_2015.parquet')
df['embedding'] = df['embedding'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df

,FieldOfStudyName,id,recovered_abstract,embedding
0,spiral,None,the utility model discloses a radiating type s...,"[-0.3100000023841858, 0.07000000029802322]"
1,fertilizer,None,the invention discloses a plant-source insect ...,"[-0.14000000059604645, -0.05000000074505806]"
2,potential flow,None,in this work we have obtained exact analytical...,"[0.05999999865889549, 0.14000000059604645]"
3,right main bronchus,None,malignant peripheral nerve sheath tumors (mpns...,"[0.15000000596046448, 0.05999999865889549]"
4,causality,None,this paper adopts a nonparametric quantile cau...,"[0.009999999776482582, -0.14000000059604645]"
...,...,...,...,...
1462051,phy,terms196,mie scattering,"[0.07000000029802322, 0.11999999731779099]"
1462052,phy,terms197,rydberg blockade,"[0.05999999865889549, 0.019999999552965164]"
1462053,phy,terms198,fabry–perot,"[0.009999999776482582, -0.10000000149011612]"
1462054,phy,terms199,raman spectroscopy,"[0.20999999344348907, 0.1599999964237213]"


In [4]:
import pandas as pd
import numpy as np
from itertools import combinations
from tqdm import tqdm

highlight_terms = [f'terms{i}' for i in range(1, 201)]
highlight_df = df[df['id'].isin(highlight_terms)].copy()
highlight_df['embedding'] = highlight_df['embedding'].apply(np.array)
highlight_df = highlight_df[highlight_df['embedding'].apply(lambda x: len(x) == len(highlight_df.iloc[0]['embedding']))].copy()

results = []
for (idx1, row1), (idx2, row2) in tqdm(combinations(highlight_df.iterrows(), 2), total=19900):  # C(200,2) = 19900
    if row1['FieldOfStudyName'] == row2['FieldOfStudyName']:
        continue  
    sim = np.linalg.norm(row1['embedding'] - row2['embedding'])
    results.append({
        'id1': row1['id'],
        'id2': row2['id'],
        'FieldOfStudyName1': row1['FieldOfStudyName'],  
        'FieldOfStudyName2': row2['FieldOfStudyName'],  
        'distance': sim
    })

similarity_df = pd.DataFrame(results)
similarity_df.to_csv('diff_field_distance.csv', index=False)

100%|██████████| 19900/19900 [00:01<00:00, 16375.77it/s]


In [14]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from itertools import combinations
from tqdm import tqdm
highlight_terms = [f'terms{i}' for i in range(1, 201)]
highlight_df = df[df['id'].isin(highlight_terms)].copy()
highlight_df['embedding'] = highlight_df['embedding'].apply(np.array)  
highlight_df = highlight_df[highlight_df['embedding'].apply(lambda x: len(x) == len(highlight_df.iloc[0]['embedding']))].copy()
results = []
for (idx1, row1), (idx2, row2) in tqdm(combinations(highlight_df.iterrows(), 2), total=19900):  # C(200,2) = 19900
    if row1['FieldOfStudyName'] == row2['FieldOfStudyName']:
        continue  

    sim = cosine_similarity(row1['embedding'].reshape(1, -1), row2['embedding'].reshape(1, -1))[0][0]
    results.append({
        'id1': row1['id'],
        'id2': row2['id'],
        'FieldOfStudyName1': row1['FieldOfStudyName'],
        'FieldOfStudyName2': row2['FieldOfStudyName'],
        'similarity': sim
    })

similarity_df = pd.DataFrame(results)

similarity_df['FieldPair'] = similarity_df.apply(
    lambda row: tuple(sorted([row['FieldOfStudyName1'], row['FieldOfStudyName2']])), axis=1
)
similarity_df.to_csv('pairwise.csv',index =False)
top_k_per_fieldpair = (
    similarity_df
    .sort_values(by='similarity')  
    .groupby('FieldPair')
    .head(500)  
    .reset_index(drop=True)
)
top_k_per_fieldpair.to_csv('data.csv', index=False)

100%|██████████| 19900/19900 [00:08<00:00, 2256.88it/s]


In [4]:
import pandas as pd
df=pd.read_csv('pairwise.csv')
df

,id1,id2,FieldOfStudyName1,FieldOfStudyName2,similarity_ids,FieldPair,paper1,paper2,sim_id1_paper1,sim_id2_paper2,sim_paper1_paper2
0,terms1,terms51,bio,comp,-0.089912,"('bio', 'comp')",1543300511,2312109180,1.0,1.0,-0.089912
1,terms1,terms52,bio,comp,0.986394,"('bio', 'comp')",1543300511,1181656224,1.0,1.0,0.986394
2,terms1,terms53,bio,comp,-0.460317,"('bio', 'comp')",1543300511,3099215994,1.0,1.0,-0.460317
3,terms1,terms54,bio,comp,-0.999168,"('bio', 'comp')",1543300511,2870319626,1.0,1.0,-0.999168
4,terms1,terms55,bio,comp,-0.757098,"('bio', 'comp')",1543300511,2513371835,1.0,1.0,-0.757098
...,...,...,...,...,...,...,...,...,...,...,...
14995,terms150,terms196,soc,phy,-0.902639,"('phy', 'soc')",2333203316,3141093539,1.0,1.0,-0.902639
14996,terms150,terms197,soc,phy,-0.393919,"('phy', 'soc')",2333203316,2322395374,1.0,1.0,-0.393919
14997,terms150,terms198,soc,phy,0.983337,"('phy', 'soc')",2333203316,2044816334,1.0,1.0,0.983337
14998,terms150,terms199,soc,phy,-0.670007,"('phy', 'soc')",2333203316,2477124584,1.0,1.0,-0.670007


In [5]:
similarity_df=df
similarity_df['FieldPair'] = similarity_df.apply(
    lambda row: tuple(sorted([row['FieldOfStudyName1'], row['FieldOfStudyName2']])), axis=1
)
low_sim_df = similarity_df[similarity_df['sim_paper1_paper2'] < 0.1]
fieldpair_distribution = (
    low_sim_df['FieldPair']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'FieldPair', 'FieldPair': 'count'})
)
fieldpair_distribution

,count,count
0,"(phy, soc)",2022
1,"(bio, comp)",1969
2,"(bio, soc)",1883
3,"(comp, phy)",1793
4,"(bio, phy)",1058
5,"(comp, soc)",544


In [20]:
import pandas as pd
import numpy as np
df = pd.read_csv('same_field_distance.csv')
eligible_fields = df['FieldOfStudyName'].value_counts()
eligible_fields = eligible_fields[eligible_fields >= 750].index
df = df[df['FieldOfStudyName'].isin(eligible_fields)]
means = []
for _ in range(100):
    sampled = (
        df.groupby('FieldOfStudyName')
        .sample(n=750, random_state=np.random.randint(0, 100000))
    )
    means.append(sampled['distance'].mean())
result_df = pd.DataFrame({'run': list(range(1, 101)), 'mean_distance': means})
result_df['mean_distance'].mean()

np.float64(0.12824057284688675)

In [6]:
# First, ensure 'FieldPair' column exists in the full df
similarity_df['FieldPair'] = similarity_df.apply(
    lambda row: tuple(sorted([row['FieldOfStudyName1'], row['FieldOfStudyName2']])), axis=1
)

# Filter for low similarity
low_sim_df = similarity_df[similarity_df['sim_paper1_paper2'] < 0.1]

# Sort within each FieldPair by similarity and take smallest 500 per group
smallest_500_per_fieldpair = (
    low_sim_df
    .sort_values(['FieldPair', 'sim_paper1_paper2'])
    .groupby('FieldPair', group_keys=False)
    .head(500)
)
smallest_500_per_fieldpair.to_csv('diff.csv')

In [7]:
import pandas as pd
df = pd.read_csv('diff.csv')
df2 = pd.read_csv('/home/honglinbao/chain_of_hints/embedding/used_data/diff_field_distance.csv')
df['pair'] = df.apply(lambda row: tuple(sorted([row['id1'], row['id2']])), axis=1)
df2['pair'] = df2.apply(lambda row: tuple(sorted([row['id1'], row['id2']])), axis=1)
df2 = df2[['pair', 'distance']]
df = df.merge(df2, on='pair', how='left')
df = df.drop(columns=['pair'])
df.to_csv('diff.csv',index=False)

In [6]:
'''
import pandas as pd
import matplotlib.pyplot as plt
df['x'] = df['embedding'].apply(lambda v: round(v[0], 2))
df['y'] = df['embedding'].apply(lambda v: round(v[1], 2))
highlight_ids = {f'terms{i}' for i in range(1, 201)}
df['highlight'] = df['id'].isin(highlight_ids)
plt.figure(figsize=(10, 8))
plt.scatter(df[~df['highlight']]['x'], df[~df['highlight']]['y'], s=1, color='gray', alpha=0.5)
plt.scatter(df[df['highlight']]['x'], df[df['highlight']]['y'], s=3, color='red', label='selected concepts')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
'''

"\nimport pandas as pd\nimport matplotlib.pyplot as plt\ndf['x'] = df['embedding'].apply(lambda v: round(v[0], 2))\ndf['y'] = df['embedding'].apply(lambda v: round(v[1], 2))\nhighlight_ids = {f'terms{i}' for i in range(1, 201)}\ndf['highlight'] = df['id'].isin(highlight_ids)\nplt.figure(figsize=(10, 8))\nplt.scatter(df[~df['highlight']]['x'], df[~df['highlight']]['y'], s=1, color='gray', alpha=0.5)\nplt.scatter(df[df['highlight']]['x'], df[df['highlight']]['y'], s=3, color='red', label='selected concepts')\nplt.xlabel('x')\nplt.ylabel('y')\nplt.legend()\nplt.grid(True)\nplt.tight_layout()\nplt.show()\n"

In [8]:
import pandas as pd
df = pd.read_csv('diff.csv')
df2 = pd.read_parquet('/net/scratch/honglinbao/coh_2015.parquet')
df['paper1'] = df['paper1'].astype(str)
df['paper2'] = df['paper2'].astype(str)
df2['id'] = df2['id'].astype(str)
# Merge to get abstract for paper1
df = df.merge(
    df2[['id', 'recovered_abstract']].rename(columns={'id': 'paper1', 'recovered_abstract': 'abstract1'}),
    on='paper1',
    how='left'
)

# Merge to get abstract for paper2
df = df.merge(
    df2[['id', 'recovered_abstract']].rename(columns={'id': 'paper2', 'recovered_abstract': 'abstract2'}),
    on='paper2',
    how='left'
)
df.to_csv('diff.csv',index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'diff.csv'